In [5]:
"""
ULTRA-FAST IDS SIMULATION - JSONL FORMAT
Features: Append-only writes, streaming-friendly

Author: Tan Yi Feng
Student ID: 23WMR14766
"""
import pandas as pd
import numpy as np
import joblib
import yaml
import sys
import pickle
import logging
import time
import json
import warnings
import os
import threading
import psutil
from pathlib import Path
from datetime import datetime
from collections import deque
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    classification_report, accuracy_score,
    precision_recall_fscore_support, confusion_matrix
)


warnings.filterwarnings('ignore', message='X does not have valid feature names')
from pathlib import Path
import joblib
import pickle
import pandas as pd


SCRIPT_DIR = Path.cwd()


DATASET_PATH = SCRIPT_DIR / 'data' / 'processed' / 'full_inference_cleaned_casing_preserved.csv'
CONFIG_PATH = SCRIPT_DIR / 'config' / 'config.yaml'
MODELS_DIR = SCRIPT_DIR / 'data' / 'models'
SRC_PATH = SCRIPT_DIR / 'src'

sys.path.append(str(SRC_PATH))

from preprocessing.feature_extractor import FeatureExtractor


logging.basicConfig(
    level=logging.WARNING,
    format='%(asctime)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger(__name__)


progress_logger = logging.getLogger('progress')
progress_logger.setLevel(logging.INFO)
console_handler = logging.StreamHandler()
console_handler.setFormatter(logging.Formatter('%(message)s'))
progress_logger.addHandler(console_handler)


ATTACK_LABELS = {
    0: "Benign (Normal)",
    1: "ARP Spoofing", 2: "MQTT Connect Flood", 3: "MQTT Publish Flood",
    4: "MQTT Malformed", 5: "Reconnaissance", 6: "Recon (VulnScan)",
    7: "ICMP Flood", 8: "SYN Flood", 9: "TCP Flood", 10: "UDP Flood",
    100: "Ambiguous"
}

ATTACK_SEVERITY = {
    0: "INFO", 1: "MEDIUM", 2: "CRITICAL", 3: "CRITICAL",
    4: "HIGH", 5: "MEDIUM", 6: "MEDIUM", 7: "CRITICAL",
    8: "CRITICAL", 9: "CRITICAL", 10: "CRITICAL", 100: "LOW"
}

ATTACK_DESCRIPTIONS = {
    1: "ARP cache poisoning detected - potential MITM attack",
    2: "MQTT connection flood - service disruption attempt",
    3: "MQTT publish flood - broker overload attack",
    4: "Malformed MQTT packets - protocol exploitation attempt",
    5: "Network reconnaissance - information gathering phase",
    6: "Vulnerability scanning - automated probing detected",
    7: "ICMP flood - network saturation attack",
    8: "SYN flood - TCP handshake exhaustion attack",
    9: "TCP flood - connection-based DoS attack",
    10: "UDP flood - bandwidth exhaustion attack",
    100: "Uncertain threat - requires manual investigation"
}

def load_config(path):
    with open(path, 'r', encoding='utf-8') as f:
        return yaml.safe_load(f)


class ResourceMonitor:
    """
    Continuously samples CPU % and RSS memory in a background thread.
    CPU % is normalized to 0-100% scale (percentage of total CPU capacity).
    Call start() before processing, stop() after.
    """
    def __init__(self, interval=0.5):
        self.interval = interval
        self.process = psutil.Process(os.getpid())
        self.num_cpus = psutil.cpu_count(logical=True)
        self._stop_event = threading.Event()
        self._thread = None

        # Metrics storage
        self.cpu_samples = []
        self.mem_samples_mb = []
        self.peak_cpu = 0.0
        self.peak_mem_mb = 0.0

    def _sample_loop(self):
        
        self.process.cpu_percent(interval=None)
        while not self._stop_event.is_set():
            try:
                
                cpu = self.process.cpu_percent(interval=None) / self.num_cpus
                mem_mb = self.process.memory_info().rss / (1024 ** 2)

                self.cpu_samples.append(cpu)
                self.mem_samples_mb.append(mem_mb)

                if cpu > self.peak_cpu:
                    self.peak_cpu = cpu
                if mem_mb > self.peak_mem_mb:
                    self.peak_mem_mb = mem_mb
            except (psutil.NoSuchProcess, psutil.AccessDenied):
                break
            time.sleep(self.interval)

    def start(self):
        self._stop_event.clear()
        self._thread = threading.Thread(target=self._sample_loop, daemon=True)
        self._thread.start()

    def stop(self):
        self._stop_event.set()
        if self._thread:
            self._thread.join(timeout=2)

    def snapshot(self):
        """Return normalized CPU % and memory MB (point-in-time)."""
        try:
            cpu = self.process.cpu_percent(interval=None) / self.num_cpus
            mem_mb = self.process.memory_info().rss / (1024 ** 2)
            return cpu, mem_mb
        except (psutil.NoSuchProcess, psutil.AccessDenied):
            return 0.0, 0.0

    @property
    def avg_cpu(self):
        return float(np.mean(self.cpu_samples)) if self.cpu_samples else 0.0

    @property
    def avg_mem_mb(self):
        return float(np.mean(self.mem_samples_mb)) if self.mem_samples_mb else 0.0


class AlertManager:
    """
    JSONL format for maximum performance:
    - No reading required (append-only)
    - Each line is a complete JSON object
    - Perfect for streaming/tailing
    """
    def __init__(self, jsonl_file='alerts.jsonl'):
        self.jsonl_file = jsonl_file
        self.alert_counts = {severity: 0 for severity in set(ATTACK_SEVERITY.values())}
        self.attack_counts = {label: 0 for label in ATTACK_LABELS.keys()}
        self.file_handle = open(self.jsonl_file, 'w', buffering=1)

    def save_alert(self, attack_id, confidence, features_dict, packet_index=None):
        timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        severity = ATTACK_SEVERITY.get(attack_id, "UNKNOWN")
        attack_name = ATTACK_LABELS.get(attack_id, f"Unknown-{attack_id}")
        description = ATTACK_DESCRIPTIONS.get(attack_id, "Unknown threat detected")

        self.alert_counts[severity] = self.alert_counts.get(severity, 0) + 1
        self.attack_counts[attack_id] = self.attack_counts.get(attack_id, 0) + 1

        alert = {
            "id": int(packet_index) if packet_index is not None else None,
            "timestamp": timestamp,
            "severity": severity,
            "attack_type": attack_name,
            "attack_id": int(attack_id),
            "description": description,
            "confidence": round(float(confidence), 4),
            "features": features_dict
        }

        try:
            self.file_handle.write(json.dumps(alert) + '\n')
        except Exception as e:
            logger.error(f"JSONL Write Error: {e}")

    def close(self):
        if self.file_handle and not self.file_handle.closed:
            self.file_handle.close()

    def print_summary(self):
        print("\n" + "="*80)
        print("📊 ALERT SUMMARY")
        print("="*80)

        print("\nBy Severity Level:")
        for severity in ['CRITICAL', 'HIGH', 'MEDIUM', 'LOW', 'INFO']:
            count = self.alert_counts.get(severity, 0)
            if count > 0:
                print(f"  {severity:>8}: {count:>8,} alerts")

        print("\nBy Attack Type:")
        for attack_id, count in sorted(self.attack_counts.items(), key=lambda x: x[1], reverse=True):
            if count > 0 and attack_id != 0:
                name = ATTACK_LABELS.get(attack_id, f"Unknown-{attack_id}")
                severity = ATTACK_SEVERITY.get(attack_id, "UNKNOWN")
                print(f"  [{severity:>8}] {name:<25}: {count:>8,}")


class RealTimeIDS:
    def __init__(self, models_dir, config):
        self.config = config
        self.alert_manager = AlertManager()
        self.resource_monitor = ResourceMonitor(interval=0.5)

        progress_logger.info("Loading IDS models...")
        self.ensemble = joblib.load(models_dir / 'ensemble.joblib')
        self.scaler = joblib.load(models_dir / 'scaler.joblib')
        self.rf_pipeline = joblib.load(models_dir / 'best_rf_pipeline2.pkl')

        with open(models_dir / 'selected_features2.pkl', 'rb') as f:
            self.selected_features = list(pickle.load(f))

        self.feature_extractor = FeatureExtractor()
        self.threshold = config.get('ensemble', {}).get('threshold_low', 0.1)

        # Stats
        self.total_packets = 0
        self.layer1_suspicious = 0
        self.layer2_attacks = 0
        self.processing_times = deque(maxlen=100)

       
        self.batch_resource_log = []

      
        self.all_y_true = []
        self.all_y_pred = []

      
        self.l1_y_true = []
        self.l1_y_pred = []

        progress_logger.info("✅ Models loaded successfully")

    def start_monitoring(self):
        self.resource_monitor.start()

    def stop_monitoring(self):
        self.resource_monitor.stop()

    def process_batch(self, batch_df, batch_num, total_batches):
        batch_start = time.time()
        batch_size = len(batch_df)

        progress_logger.info(f"[Batch {batch_num}/{total_batches}] Processing {batch_size} packets...")

       
        features_df = self.feature_extractor.extract_from_dataframe(batch_df)
        features_norm, _ = self.feature_extractor.normalize_features(features_df, self.scaler)
        X_L1 = self.feature_extractor.get_feature_vector(features_norm)

        predictions = self.ensemble.predict_with_details(X_L1)
        layer1_flags = predictions['ensemble_score'] > self.threshold
        suspicious_indices = np.where(layer1_flags)[0]

        self.total_packets += batch_size
        self.layer1_suspicious += len(suspicious_indices)

       
        if 'label' in batch_df.columns:
            batch_y_true_raw = batch_df['label'].astype(str).str.split('.').str[0].astype(int).values
            l1_true_binary = (batch_y_true_raw != 0).astype(int)
            l1_pred_binary = layer1_flags.astype(int)
            self.l1_y_true.extend(l1_true_binary)
            self.l1_y_pred.extend(l1_pred_binary)

        batch_predictions = np.zeros(batch_size, dtype=int)


        if len(suspicious_indices) > 0:
            try:
                X_L2_subset = batch_df.iloc[suspicious_indices][self.selected_features].copy()
                X_L2_subset.replace([np.inf, -np.inf], np.nan, inplace=True)

                imputer = SimpleImputer(strategy='median')
                X_L2_clean = imputer.fit_transform(X_L2_subset)

                probs = self.rf_pipeline.predict_proba(X_L2_clean)
                max_probs = np.max(probs, axis=1)
                raw_preds = np.argmax(probs, axis=1)

                mapped_preds = raw_preds + 1  
                l2_decisions = np.where(max_probs >= 0.9, mapped_preds, 100)

                batch_predictions[suspicious_indices] = l2_decisions

                for i, (sus_idx, attack_id, confidence) in enumerate(zip(suspicious_indices, l2_decisions, max_probs)):
                    if attack_id != 0:
                        global_packet_idx = batch_df.index[sus_idx]
                        feature_dict = {}
                        packet_row = batch_df.iloc[sus_idx]

                        for feat_name in self.selected_features:
                            if feat_name in packet_row.index:
                                feat_value = packet_row[feat_name]
                                if pd.notna(feat_value) and not np.isinf(feat_value):
                                    feature_dict[feat_name] = float(feat_value)

                        self.alert_manager.save_alert(
                            attack_id=attack_id,
                            confidence=confidence,
                            features_dict=feature_dict,
                            packet_index=global_packet_idx
                        )
                        self.layer2_attacks += 1

            except Exception as e:
                logger.error(f"Layer 2 error: {e}")

       
        if 'label' in batch_df.columns:
            batch_y_true = batch_df['label'].astype(str).str.split('.').str[0].astype(int).values
            self.all_y_true.extend(batch_y_true)
            self.all_y_pred.extend(batch_predictions)

        batch_time = time.time() - batch_start
        self.processing_times.append(batch_time)

       
        cpu_snap, mem_snap = self.resource_monitor.snapshot()
        self.batch_resource_log.append((batch_num, cpu_snap, mem_snap))

        progress_logger.info(
            f"[Batch {batch_num}/{total_batches}] Complete in {batch_time:.2f}s | "
            f"Suspicious: {len(suspicious_indices)} | Total Attacks Logged: {self.layer2_attacks:,} | "
            f"CPU: {cpu_snap:.1f}% | Mem: {mem_snap:.1f} MB\n"
        )
        return batch_predictions, batch_time

    def get_stats(self):
        avg_time = np.mean(self.processing_times) if self.processing_times else 0
        return {
            'total_packets': self.total_packets,
            'layer1_suspicious': self.layer1_suspicious,
            'layer2_attacks': self.layer2_attacks,
            'avg_batch_time': avg_time,
            'avg_latency': (avg_time / 500) if avg_time > 0 else 0
        }

    def print_resource_report(self):
        mon = self.resource_monitor
        print("\n" + "="*80)
        print("🖥️  RESOURCE USAGE REPORT")
        print("="*80)
        print(f"\n  Logical CPU Cores     : {mon.num_cpus}")
        print(f"  Peak CPU Utilization  : {mon.peak_cpu:.1f}%  (of total capacity)")
        print(f"  Avg  CPU Utilization  : {mon.avg_cpu:.1f}%  (of total capacity)")
        print(f"  Peak Memory (RSS)     : {mon.peak_mem_mb:.1f} MB")
        print(f"  Avg  Memory (RSS)     : {mon.avg_mem_mb:.1f} MB")

        if self.batch_resource_log:
            print(f"\n  Per-Batch Snapshots (CPU % | Memory MB):")
            print(f"  {'Batch':>6}  {'CPU %':>7}  {'Mem MB':>8}")
            print(f"  {'-'*6}  {'-'*7}  {'-'*8}")
            for batch_num, cpu, mem in self.batch_resource_log:
                print(f"  {batch_num:>6}  {cpu:>6.1f}%  {mem:>7.1f}")

    def print_layer1_report(self):
        if not self.l1_y_true:
            print("\n⚠️ No labels found. Skipping Layer 1 report.")
            return

        y_true = np.array(self.l1_y_true)
        y_pred = np.array(self.l1_y_pred)

        print("\n" + "="*80)
        print("🔍 LAYER 1 EVALUATION (Anomaly Detector - Binary: Benign vs Attack)")
        print("="*80)

     
        precision, recall, f1, support = precision_recall_fscore_support(
            y_true, y_pred, labels=[0, 1], zero_division=0
        )

        print(f"\n  {'Class':<14} {'Precision':>10} {'Recall':>10} {'F1':>10} {'Support':>10}")
        print(f"  {'-'*56}")
        for i, label in enumerate(['Benign', 'Attack']):
            print(f"  {label:<14} {precision[i]:>9.4f}  {recall[i]:>9.4f}  {f1[i]:>9.4f}  {support[i]:>9,}")

        # Macro & weighted averages
        prec_macro, rec_macro, f1_macro, _ = precision_recall_fscore_support(
            y_true, y_pred, average='macro', zero_division=0
        )
        prec_weighted, rec_weighted, f1_weighted, _ = precision_recall_fscore_support(
            y_true, y_pred, average='weighted', zero_division=0
        )
        print(f"  {'-'*56}")
        print(f"  {'Macro Avg':<14} {prec_macro:>9.4f}  {rec_macro:>9.4f}  {f1_macro:>9.4f}  {len(y_true):>9,}")
        print(f"  {'Weighted Avg':<14} {prec_weighted:>9.4f}  {rec_weighted:>9.4f}  {f1_weighted:>9.4f}  {len(y_true):>9,}")

       
        cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
        tn, fp, fn, tp = cm.ravel()
        print(f"\n  Confusion Matrix:")
        print(f"  {'':>22} Predicted Benign  Predicted Attack")
        print(f"  {'Actual Benign':>22}       {tn:>8,}        {fp:>8,}")
        print(f"  {'Actual Attack':>22}       {fn:>8,}        {tp:>8,}")
        print(f"\n  False Positive Rate : {fp/(fp+tn)*100:.2f}%  (benign flagged as attack)")
        print(f"  False Negative Rate : {fn/(fn+tp)*100:.2f}%  (attack missed by Layer 1)")

    def print_accuracy_report(self):
        if not self.all_y_true:
            print("\n⚠️ No labels found. Skipping Layer 2 accuracy report.")
            return

        y_true = np.array(self.all_y_true)
        y_pred = np.array(self.all_y_pred)

        print("\n" + "="*80)
        print("🎯 LAYER 2 ACCURACY VERIFICATION (High Confidence Only)")
        print("="*80)

        ambiguous_count = np.sum(y_pred == 100)
        total = len(y_pred)
        print(f"\n  Ambiguous Skipped (Label 100): {ambiguous_count:,} ({ambiguous_count/total*100:.2f}%)")

        mask = y_pred != 100
        y_true_certain = y_true[mask]
        y_pred_certain = y_pred[mask]

        if len(y_true_certain) > 0:
            acc = accuracy_score(y_true_certain, y_pred_certain)
            print(f"  Accuracy (Certain)           : {acc:.4f}")
            print("\n  Detailed Classification Report (High Confidence Only):")

            unique_labels = np.unique(np.concatenate((y_true_certain, y_pred_certain)))
            labels_to_show = [l for l in unique_labels if l != 0 and l != 100]
            target_names = [ATTACK_LABELS.get(l, f"Class {l}") for l in labels_to_show]

            if labels_to_show:
                print(classification_report(
                    y_true_certain,
                    y_pred_certain,
                    labels=labels_to_show,
                    target_names=target_names,
                    digits=4,
                    zero_division=0
                ))
            else:
                print("   No attack classes found in confident predictions.")
        else:
            print("   ⚠️ No confident predictions were made.")

    def cleanup(self):
        self.alert_manager.close()
        self.resource_monitor.stop()


def run_simulation(dataset_path, packets_per_second=300, duration=None):
    print("🚀 STARTING ULTRA-FAST IDS SIMULATION")
    print(f"Output: alerts.jsonl (JSONL format - one JSON per line)")
    print("="*80)

    config = load_config(CONFIG_PATH)
    ids = RealTimeIDS(MODELS_DIR, config)

    try:
        progress_logger.info(f"Loading dataset: {dataset_path}")
        df = pd.read_csv(dataset_path)
        total_packets = len(df)

        batch_size = packets_per_second
        num_batches = (total_packets + batch_size - 1) // batch_size
        if duration:
            num_batches = min(num_batches, duration)

        print(f"Simulation Parameters:")
        print(f"  Batch Size:     {batch_size:,} packets/batch")
        print(f"  Total Batches:  {num_batches:,}")
        print("="*80 + "\n")

       
        ids.start_monitoring()
        start_time = time.time()

        for batch_num in range(1, num_batches + 1):
            start_idx = (batch_num - 1) * batch_size
            end_idx = min(start_idx + batch_size, total_packets)
            batch_df = df.iloc[start_idx:end_idx].copy()
            ids.process_batch(batch_df, batch_num, num_batches)
            time.sleep(0.05)

       
        ids.stop_monitoring()
        total_time = time.time() - start_time
        stats = ids.get_stats()

        print("\n" + "="*80)
        print("🏁 SIMULATION COMPLETE")
        print("="*80)
        print(f"\nProcessing Statistics:")
        print(f"  Total Packets   : {stats['total_packets']:,}")
        print(f"  Throughput      : {stats['total_packets']/total_time:.2f} packets/s")
        print(f"  Avg Latency     : {stats['avg_latency']*1000:.2f}ms")
        print(f"  L1 Suspicious   : {stats['layer1_suspicious']:,}")
        print(f"  L2 Attacks      : {stats['layer2_attacks']:,}")

        ids.alert_manager.print_summary()
        ids.print_resource_report()
        ids.print_layer1_report()
        ids.print_accuracy_report()

        print("\n✅ Simulation finished successfully")
        print(f"✅ Alerts saved to: {os.path.abspath('alerts.jsonl')}")

    finally:
        ids.cleanup()

if __name__ == "__main__":
    run_simulation(
        dataset_path=DATASET_PATH,
        packets_per_second=500,
        duration=100,
    )

Loading IDS models...
Loading IDS models...
Loading IDS models...
Loading IDS models...
2026-03-15 13:18:57 - INFO - Loading IDS models...


🚀 STARTING ULTRA-FAST IDS SIMULATION
Output: alerts.jsonl (JSONL format - one JSON per line)


C:\Users\tanyi\anaconda3\Lib\site-packages\sklearn\base.py:380: InconsistentVersionWarning: Trying to unpickle estimator OneClassSVM from version 1.3.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\tanyi\anaconda3\Lib\site-packages\sklearn\base.py:380: InconsistentVersionWarning: Trying to unpickle estimator LocalOutlierFactor from version 1.3.0 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\tanyi\anaconda3\Lib\site-packages\sklearn\base.py:380: InconsistentVersionWarning: Trying to unpickle estimator ExtraTreeRegressor from version 1.3.0 when using version 1.6.1. This might lead to bre

Simulation Parameters:
  Batch Size:     500 packets/batch
  Total Batches:  100



[Batch 1/100] Complete in 1.03s | Suspicious: 497 | Total Attacks Logged: 497 | CPU: 4.3% | Mem: 2446.1 MB

[Batch 1/100] Complete in 1.03s | Suspicious: 497 | Total Attacks Logged: 497 | CPU: 4.3% | Mem: 2446.1 MB

[Batch 1/100] Complete in 1.03s | Suspicious: 497 | Total Attacks Logged: 497 | CPU: 4.3% | Mem: 2446.1 MB

[Batch 1/100] Complete in 1.03s | Suspicious: 497 | Total Attacks Logged: 497 | CPU: 4.3% | Mem: 2446.1 MB

2026-03-15 13:19:42 - INFO - [Batch 1/100] Complete in 1.03s | Suspicious: 497 | Total Attacks Logged: 497 | CPU: 4.3% | Mem: 2446.1 MB

[Batch 2/100] Processing 500 packets...
[Batch 2/100] Processing 500 packets...
[Batch 2/100] Processing 500 packets...
[Batch 2/100] Processing 500 packets...
2026-03-15 13:19:42 - INFO - [Batch 2/100] Processing 500 packets...
[Batch 2/100] Complete in 0.95s | Suspicious: 498 | Total Attacks Logged: 995 | CPU: 6.8% | Mem: 2446.7 MB

[Batch 2/100] Complete in 0.95s | Suspicious: 498 | Total Attacks Logged: 995 | CPU: 6.8% | Me


🏁 SIMULATION COMPLETE

Processing Statistics:
  Total Packets   : 50,000
  Throughput      : 502.78 packets/s
  Avg Latency     : 1.87ms
  L1 Suspicious   : 49,863
  L2 Attacks      : 49,863

📊 ALERT SUMMARY

By Severity Level:
  CRITICAL:   48,662 alerts
      HIGH:       22 alerts
    MEDIUM:      764 alerts
       LOW:      415 alerts

By Attack Type:
  [CRITICAL] UDP Flood                :   15,555
  [CRITICAL] ICMP Flood               :   13,894
  [CRITICAL] SYN Flood                :    8,949
  [CRITICAL] TCP Flood                :    8,511
  [CRITICAL] MQTT Connect Flood       :    1,289
  [  MEDIUM] Reconnaissance           :      684
  [CRITICAL] MQTT Publish Flood       :      464
  [     LOW] Ambiguous                :      415
  [  MEDIUM] ARP Spoofing             :       77
  [    HIGH] MQTT Malformed           :       22
  [  MEDIUM] Recon (VulnScan)         :        3

🖥️  RESOURCE USAGE REPORT

  Logical CPU Cores     : 12
  Peak CPU Utilization  : 30.4%  (of total cap

In [8]:
"""
ULTRA-FAST IDS SIMULATION - JSONL FORMAT
Features: Append-only writes, streaming-friendly

Author: Tan Yi Feng
Student ID: 23WMR14766
"""
import pandas as pd
import numpy as np
import joblib
import yaml
import sys
import pickle
import logging
import time
import json
import warnings
import os
import threading
import psutil
from pathlib import Path
from datetime import datetime
from collections import deque
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    classification_report, accuracy_score,
    precision_recall_fscore_support, confusion_matrix
)

# Suppress warnings
warnings.filterwarnings('ignore', message='X does not have valid feature names')
from pathlib import Path
import joblib
import pickle
import pandas as pd


SCRIPT_DIR = Path.cwd()


DATASET_PATH = SCRIPT_DIR / 'data' / 'processed' / 'full_inference_cleaned_casing_preserved.csv'
CONFIG_PATH = SCRIPT_DIR / 'config' / 'config.yaml'
MODELS_DIR = SCRIPT_DIR / 'data' / 'models'
SRC_PATH = SCRIPT_DIR / 'src'

sys.path.append(str(SRC_PATH))

from preprocessing.feature_extractor import FeatureExtractor


logging.basicConfig(
    level=logging.WARNING,
    format='%(asctime)s - %(levelname)s - %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S'
)
logger = logging.getLogger(__name__)


progress_logger = logging.getLogger('progress')
progress_logger.setLevel(logging.INFO)
console_handler = logging.StreamHandler()
console_handler.setFormatter(logging.Formatter('%(message)s'))
progress_logger.addHandler(console_handler)


ATTACK_LABELS = {
    0: "Benign (Normal)",
    1: "ARP Spoofing", 2: "MQTT Connect Flood", 3: "MQTT Publish Flood",
    4: "MQTT Malformed", 5: "Reconnaissance", 6: "Recon (VulnScan)",
    7: "ICMP Flood", 8: "SYN Flood", 9: "TCP Flood", 10: "UDP Flood",
    100: "Ambiguous"
}

ATTACK_SEVERITY = {
    0: "INFO", 1: "MEDIUM", 2: "CRITICAL", 3: "CRITICAL",
    4: "HIGH", 5: "MEDIUM", 6: "MEDIUM", 7: "CRITICAL",
    8: "CRITICAL", 9: "CRITICAL", 10: "CRITICAL", 100: "LOW"
}

ATTACK_DESCRIPTIONS = {
    1: "ARP cache poisoning detected - potential MITM attack",
    2: "MQTT connection flood - service disruption attempt",
    3: "MQTT publish flood - broker overload attack",
    4: "Malformed MQTT packets - protocol exploitation attempt",
    5: "Network reconnaissance - information gathering phase",
    6: "Vulnerability scanning - automated probing detected",
    7: "ICMP flood - network saturation attack",
    8: "SYN flood - TCP handshake exhaustion attack",
    9: "TCP flood - connection-based DoS attack",
    10: "UDP flood - bandwidth exhaustion attack",
    100: "Uncertain threat - requires manual investigation"
}

def load_config(path):
    with open(path, 'r', encoding='utf-8') as f:
        return yaml.safe_load(f)


class ResourceMonitor:
    """
    Continuously samples CPU % and RSS memory in a background thread.
    CPU % is normalized to 0-100% scale (percentage of total CPU capacity).
    Call start() before processing, stop() after.
    """
    def __init__(self, interval=0.5):
        self.interval = interval
        self.process = psutil.Process(os.getpid())
        self.num_cpus = psutil.cpu_count(logical=True)
        self._stop_event = threading.Event()
        self._thread = None

       
        self.cpu_samples = []
        self.mem_samples_mb = []
        self.peak_cpu = 0.0
        self.peak_mem_mb = 0.0

    def _sample_loop(self):
        
        self.process.cpu_percent(interval=None)
        while not self._stop_event.is_set():
            try:
                
                cpu = self.process.cpu_percent(interval=None) / self.num_cpus
                mem_mb = self.process.memory_info().rss / (1024 ** 2)

                self.cpu_samples.append(cpu)
                self.mem_samples_mb.append(mem_mb)

                if cpu > self.peak_cpu:
                    self.peak_cpu = cpu
                if mem_mb > self.peak_mem_mb:
                    self.peak_mem_mb = mem_mb
            except (psutil.NoSuchProcess, psutil.AccessDenied):
                break
            time.sleep(self.interval)

    def start(self):
        self._stop_event.clear()
        self._thread = threading.Thread(target=self._sample_loop, daemon=True)
        self._thread.start()

    def stop(self):
        self._stop_event.set()
        if self._thread:
            self._thread.join(timeout=2)

    def snapshot(self):
        """Return normalized CPU % and memory MB (point-in-time)."""
        try:
            cpu = self.process.cpu_percent(interval=None) / self.num_cpus
            mem_mb = self.process.memory_info().rss / (1024 ** 2)
            return cpu, mem_mb
        except (psutil.NoSuchProcess, psutil.AccessDenied):
            return 0.0, 0.0

    @property
    def avg_cpu(self):
        return float(np.mean(self.cpu_samples)) if self.cpu_samples else 0.0

    @property
    def avg_mem_mb(self):
        return float(np.mean(self.mem_samples_mb)) if self.mem_samples_mb else 0.0


class AlertManager:
    """
    JSONL format for maximum performance:
    - No reading required (append-only)
    - Each line is a complete JSON object
    - Perfect for streaming/tailing
    """
    def __init__(self, jsonl_file='alerts.jsonl'):
        self.jsonl_file = jsonl_file
        self.alert_counts = {severity: 0 for severity in set(ATTACK_SEVERITY.values())}
        self.attack_counts = {label: 0 for label in ATTACK_LABELS.keys()}
        self.file_handle = open(self.jsonl_file, 'w', buffering=1)

    def save_alert(self, attack_id, confidence, features_dict, packet_index=None):
        timestamp = datetime.now().strftime('%Y-%m-%d %H:%M:%S')
        severity = ATTACK_SEVERITY.get(attack_id, "UNKNOWN")
        attack_name = ATTACK_LABELS.get(attack_id, f"Unknown-{attack_id}")
        description = ATTACK_DESCRIPTIONS.get(attack_id, "Unknown threat detected")

        self.alert_counts[severity] = self.alert_counts.get(severity, 0) + 1
        self.attack_counts[attack_id] = self.attack_counts.get(attack_id, 0) + 1

        alert = {
            "id": int(packet_index) if packet_index is not None else None,
            "timestamp": timestamp,
            "severity": severity,
            "attack_type": attack_name,
            "attack_id": int(attack_id),
            "description": description,
            "confidence": round(float(confidence), 4),
            "features": features_dict
        }

        try:
            self.file_handle.write(json.dumps(alert) + '\n')
        except Exception as e:
            logger.error(f"JSONL Write Error: {e}")

    def close(self):
        if self.file_handle and not self.file_handle.closed:
            self.file_handle.close()

    def print_summary(self):
        print("\n" + "="*80)
        print("📊 ALERT SUMMARY")
        print("="*80)

        print("\nBy Severity Level:")
        for severity in ['CRITICAL', 'HIGH', 'MEDIUM', 'LOW', 'INFO']:
            count = self.alert_counts.get(severity, 0)
            if count > 0:
                print(f"  {severity:>8}: {count:>8,} alerts")

        print("\nBy Attack Type:")
        for attack_id, count in sorted(self.attack_counts.items(), key=lambda x: x[1], reverse=True):
            if count > 0 and attack_id != 0:
                name = ATTACK_LABELS.get(attack_id, f"Unknown-{attack_id}")
                severity = ATTACK_SEVERITY.get(attack_id, "UNKNOWN")
                print(f"  [{severity:>8}] {name:<25}: {count:>8,}")


class RealTimeIDS:
    def __init__(self, models_dir, config):
        self.config = config
        self.alert_manager = AlertManager()
        self.resource_monitor = ResourceMonitor(interval=0.5)

        progress_logger.info("Loading IDS models...")
        self.ensemble = joblib.load(models_dir / 'ensemble.joblib')
        self.scaler = joblib.load(models_dir / 'scaler.joblib')
        self.rf_pipeline = joblib.load(models_dir / 'best_rf_pipeline2.pkl')

        with open(models_dir / 'selected_features2.pkl', 'rb') as f:
            self.selected_features = list(pickle.load(f))

        self.feature_extractor = FeatureExtractor()
        self.threshold = config.get('ensemble', {}).get('threshold_low', 0.1)

        # Stats
        self.total_packets = 0
        self.layer1_suspicious = 0
        self.layer2_attacks = 0
        self.processing_times = deque(maxlen=100)

       
        self.batch_resource_log = []

        self.all_y_true = []
        self.all_y_pred = []

        self.l1_y_true = []
        self.l1_y_pred = []

        progress_logger.info("✅ Models loaded successfully")

    def start_monitoring(self):
        self.resource_monitor.start()

    def stop_monitoring(self):
        self.resource_monitor.stop()

    def process_batch(self, batch_df, batch_num, total_batches):
        batch_start = time.time()
        batch_size = len(batch_df)

        progress_logger.info(f"[Batch {batch_num}/{total_batches}] Processing {batch_size} packets...")

       
        features_df = self.feature_extractor.extract_from_dataframe(batch_df)
        features_norm, _ = self.feature_extractor.normalize_features(features_df, self.scaler)
        X_L1 = self.feature_extractor.get_feature_vector(features_norm)

        predictions = self.ensemble.predict_with_details(X_L1)
        layer1_flags = predictions['ensemble_score'] > self.threshold
        suspicious_indices = np.where(layer1_flags)[0]

        self.total_packets += batch_size
        self.layer1_suspicious += len(suspicious_indices)

        
        if 'label' in batch_df.columns:
            batch_y_true_raw = batch_df['label'].astype(str).str.split('.').str[0].astype(int).values
            l1_true_binary = (batch_y_true_raw != 0).astype(int)
            l1_pred_binary = layer1_flags.astype(int)
            self.l1_y_true.extend(l1_true_binary)
            self.l1_y_pred.extend(l1_pred_binary)

        batch_predictions = np.zeros(batch_size, dtype=int)

       
        if len(suspicious_indices) > 0:
            try:
                X_L2_subset = batch_df.iloc[suspicious_indices][self.selected_features].copy()
                X_L2_subset.replace([np.inf, -np.inf], np.nan, inplace=True)

                imputer = SimpleImputer(strategy='median')
                X_L2_clean = imputer.fit_transform(X_L2_subset)

                probs = self.rf_pipeline.predict_proba(X_L2_clean)
                max_probs = np.max(probs, axis=1)
                raw_preds = np.argmax(probs, axis=1)

                mapped_preds = raw_preds + 1  # Correction for LabelEncoder
                l2_decisions = np.where(max_probs >= 0.9, mapped_preds, 100)

                batch_predictions[suspicious_indices] = l2_decisions

                for i, (sus_idx, attack_id, confidence) in enumerate(zip(suspicious_indices, l2_decisions, max_probs)):
                    if attack_id != 0:
                        global_packet_idx = batch_df.index[sus_idx]
                        feature_dict = {}
                        packet_row = batch_df.iloc[sus_idx]

                        for feat_name in self.selected_features:
                            if feat_name in packet_row.index:
                                feat_value = packet_row[feat_name]
                                if pd.notna(feat_value) and not np.isinf(feat_value):
                                    feature_dict[feat_name] = float(feat_value)

                        self.alert_manager.save_alert(
                            attack_id=attack_id,
                            confidence=confidence,
                            features_dict=feature_dict,
                            packet_index=global_packet_idx
                        )
                        self.layer2_attacks += 1

            except Exception as e:
                logger.error(f"Layer 2 error: {e}")

       
        if 'label' in batch_df.columns:
            batch_y_true = batch_df['label'].astype(str).str.split('.').str[0].astype(int).values
            self.all_y_true.extend(batch_y_true)
            self.all_y_pred.extend(batch_predictions)

        batch_time = time.time() - batch_start
        self.processing_times.append(batch_time)

       
        cpu_snap, mem_snap = self.resource_monitor.snapshot()
        self.batch_resource_log.append((batch_num, cpu_snap, mem_snap))

        progress_logger.info(
            f"[Batch {batch_num}/{total_batches}] Complete in {batch_time:.2f}s | "
            f"Suspicious: {len(suspicious_indices)} | Total Attacks Logged: {self.layer2_attacks:,} | "
            f"CPU: {cpu_snap:.1f}% | Mem: {mem_snap:.1f} MB\n"
        )
        return batch_predictions, batch_time

    def get_stats(self):
        avg_time = np.mean(self.processing_times) if self.processing_times else 0
        return {
            'total_packets': self.total_packets,
            'layer1_suspicious': self.layer1_suspicious,
            'layer2_attacks': self.layer2_attacks,
            'avg_batch_time': avg_time,
            'avg_latency': (avg_time / 500) if avg_time > 0 else 0
        }

    def print_resource_report(self):
        mon = self.resource_monitor
        print("\n" + "="*80)
        print("🖥️  RESOURCE USAGE REPORT")
        print("="*80)
        print(f"\n  Logical CPU Cores     : {mon.num_cpus}")
        print(f"  Peak CPU Utilization  : {mon.peak_cpu:.1f}%  (of total capacity)")
        print(f"  Avg  CPU Utilization  : {mon.avg_cpu:.1f}%  (of total capacity)")
        print(f"  Peak Memory (RSS)     : {mon.peak_mem_mb:.1f} MB")
        print(f"  Avg  Memory (RSS)     : {mon.avg_mem_mb:.1f} MB")

        if self.batch_resource_log:
            print(f"\n  Per-Batch Snapshots (CPU % | Memory MB):")
            print(f"  {'Batch':>6}  {'CPU %':>7}  {'Mem MB':>8}")
            print(f"  {'-'*6}  {'-'*7}  {'-'*8}")
            for batch_num, cpu, mem in self.batch_resource_log:
                print(f"  {batch_num:>6}  {cpu:>6.1f}%  {mem:>7.1f}")

    def print_layer1_report(self):
        if not self.l1_y_true:
            print("\n⚠️ No labels found. Skipping Layer 1 report.")
            return

        y_true = np.array(self.l1_y_true)
        y_pred = np.array(self.l1_y_pred)

        print("\n" + "="*80)
        print("🔍 LAYER 1 EVALUATION (Anomaly Detector - Binary: Benign vs Attack)")
        print("="*80)

        # Per-class metrics
        precision, recall, f1, support = precision_recall_fscore_support(
            y_true, y_pred, labels=[0, 1], zero_division=0
        )

        print(f"\n  {'Class':<14} {'Precision':>10} {'Recall':>10} {'F1':>10} {'Support':>10}")
        print(f"  {'-'*56}")
        for i, label in enumerate(['Benign', 'Attack']):
            print(f"  {label:<14} {precision[i]:>9.4f}  {recall[i]:>9.4f}  {f1[i]:>9.4f}  {support[i]:>9,}")

      
        prec_macro, rec_macro, f1_macro, _ = precision_recall_fscore_support(
            y_true, y_pred, average='macro', zero_division=0
        )
        prec_weighted, rec_weighted, f1_weighted, _ = precision_recall_fscore_support(
            y_true, y_pred, average='weighted', zero_division=0
        )
        print(f"  {'-'*56}")
        print(f"  {'Macro Avg':<14} {prec_macro:>9.4f}  {rec_macro:>9.4f}  {f1_macro:>9.4f}  {len(y_true):>9,}")
        print(f"  {'Weighted Avg':<14} {prec_weighted:>9.4f}  {rec_weighted:>9.4f}  {f1_weighted:>9.4f}  {len(y_true):>9,}")

       
        cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
        tn, fp, fn, tp = cm.ravel()
        print(f"\n  Confusion Matrix:")
        print(f"  {'':>22} Predicted Benign  Predicted Attack")
        print(f"  {'Actual Benign':>22}       {tn:>8,}        {fp:>8,}")
        print(f"  {'Actual Attack':>22}       {fn:>8,}        {tp:>8,}")
        print(f"\n  False Positive Rate : {fp/(fp+tn)*100:.2f}%  (benign flagged as attack)")
        print(f"  False Negative Rate : {fn/(fn+tp)*100:.2f}%  (attack missed by Layer 1)")

    def print_accuracy_report(self):
        if not self.all_y_true:
            print("\n⚠️ No labels found. Skipping Layer 2 accuracy report.")
            return

        y_true = np.array(self.all_y_true)
        y_pred = np.array(self.all_y_pred)

        print("\n" + "="*80)
        print("🎯 LAYER 2 ACCURACY VERIFICATION (High Confidence Only)")
        print("="*80)

        ambiguous_count = np.sum(y_pred == 100)
        total = len(y_pred)
        print(f"\n  Ambiguous Skipped (Label 100): {ambiguous_count:,} ({ambiguous_count/total*100:.2f}%)")

        mask = y_pred != 100
        y_true_certain = y_true[mask]
        y_pred_certain = y_pred[mask]

        if len(y_true_certain) > 0:
            acc = accuracy_score(y_true_certain, y_pred_certain)
            print(f"  Accuracy (Certain)           : {acc:.4f}")
            print("\n  Detailed Classification Report (High Confidence Only):")

            unique_labels = np.unique(np.concatenate((y_true_certain, y_pred_certain)))
            labels_to_show = [l for l in unique_labels if l != 0 and l != 100]
            target_names = [ATTACK_LABELS.get(l, f"Class {l}") for l in labels_to_show]

            if labels_to_show:
                print(classification_report(
                    y_true_certain,
                    y_pred_certain,
                    labels=labels_to_show,
                    target_names=target_names,
                    digits=4,
                    zero_division=0
                ))
            else:
                print("   No attack classes found in confident predictions.")
        else:
            print("   ⚠️ No confident predictions were made.")

    def cleanup(self):
        self.alert_manager.close()
        self.resource_monitor.stop()

def run_simulation(dataset_path, packets_per_second=300, duration=None):
    print("🚀 STARTING ULTRA-FAST IDS SIMULATION")
    print(f"Output: alerts.jsonl (JSONL format - one JSON per line)")
    print("="*80)

    config = load_config(CONFIG_PATH)
    ids = RealTimeIDS(MODELS_DIR, config)

    try:
        progress_logger.info(f"Loading dataset: {dataset_path}")
        df = pd.read_csv(dataset_path)
        total_packets = len(df)

        batch_size = packets_per_second
        num_batches = (total_packets + batch_size - 1) // batch_size
        if duration:
            num_batches = min(num_batches, duration)

        print(f"Simulation Parameters:")
        print(f"  Batch Size:     {batch_size:,} packets/batch")
        print(f"  Total Batches:  {num_batches:,}")
        print("="*80 + "\n")

       
        ids.start_monitoring()
        start_time = time.time()

        for batch_num in range(1, num_batches + 1):
            start_idx = (batch_num - 1) * batch_size
            end_idx = min(start_idx + batch_size, total_packets)
            batch_df = df.iloc[start_idx:end_idx].copy()
            ids.process_batch(batch_df, batch_num, num_batches)
            time.sleep(0.05)

        
        ids.stop_monitoring()
        total_time = time.time() - start_time
        stats = ids.get_stats()

        print("\n" + "="*80)
        print("🏁 SIMULATION COMPLETE")
        print("="*80)
        print(f"\nProcessing Statistics:")
        print(f"  Total Packets   : {stats['total_packets']:,}")
        print(f"  Throughput      : {stats['total_packets']/total_time:.2f} packets/s")
        print(f"  Avg Latency     : {stats['avg_latency']*1000:.2f}ms")
        print(f"  L1 Suspicious   : {stats['layer1_suspicious']:,}")
        print(f"  L2 Attacks      : {stats['layer2_attacks']:,}")

        ids.alert_manager.print_summary()
        ids.print_resource_report()
        ids.print_layer1_report()
        ids.print_accuracy_report()

        print("\n✅ Simulation finished successfully")
        print(f"✅ Alerts saved to: {os.path.abspath('alerts.jsonl')}")

    finally:
        ids.cleanup()

if __name__ == "__main__":
    run_simulation(
        dataset_path=DATASET_PATH,
        packets_per_second=500,
        duration=100,
    )

Loading IDS models...
Loading IDS models...
Loading IDS models...
Loading IDS models...
Loading IDS models...
Loading IDS models...
Loading IDS models...
2026-03-20 17:37:14 - INFO - Loading IDS models...


🚀 STARTING ULTRA-FAST IDS SIMULATION
Output: alerts.jsonl (JSONL format - one JSON per line)


✅ Models loaded successfully
✅ Models loaded successfully
✅ Models loaded successfully
✅ Models loaded successfully
✅ Models loaded successfully
✅ Models loaded successfully
✅ Models loaded successfully
2026-03-20 17:37:14 - INFO - ✅ Models loaded successfully
Loading dataset: C:\Users\tanyi\Downloads\iomt_ids (2)\iomt_ids\data\processed\full_inference_cleaned_casing_preserved.csv
Loading dataset: C:\Users\tanyi\Downloads\iomt_ids (2)\iomt_ids\data\processed\full_inference_cleaned_casing_preserved.csv
Loading dataset: C:\Users\tanyi\Downloads\iomt_ids (2)\iomt_ids\data\processed\full_inference_cleaned_casing_preserved.csv
Loading dataset: C:\Users\tanyi\Downloads\iomt_ids (2)\iomt_ids\data\processed\full_inference_cleaned_casing_preserved.csv
Loading dataset: C:\Users\tanyi\Downloads\iomt_ids (2)\iomt_ids\data\processed\full_inference_cleaned_casing_preserved.csv
Loading dataset: C:\Users\tanyi\Downloads\iomt_ids (2)\iomt_ids\data\processed\full_inference_cleaned_casing_preserved.csv
L

Simulation Parameters:
  Batch Size:     500 packets/batch
  Total Batches:  100



C:\Users\tanyi\anaconda3\Lib\site-packages\sklearn\utils\validation.py:2732: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
[Batch 1/100] Complete in 1.22s | Suspicious: 497 | Total Attacks Logged: 497 | CPU: 7.9% | Mem: 2484.6 MB

[Batch 1/100] Complete in 1.22s | Suspicious: 497 | Total Attacks Logged: 497 | CPU: 7.9% | Mem: 2484.6 MB

[Batch 1/100] Complete in 1.22s | Suspicious: 497 | Total Attacks Logged: 497 | CPU: 7.9% | Mem: 2484.6 MB

[Batch 1/100] Complete in 1.22s | Suspicious: 497 | Total Attacks Logged: 497 | CPU: 7.9% | Mem: 2484.6 MB

[Batch 1/100] Complete in 1.22s | Suspicious: 497 | Total Attacks Logged: 497 | CPU: 7.9% | Mem: 2484.6 MB

[Batch 1/100] Complete in 1.22s | Suspicious: 497 | Total Attacks Logged: 497 | CPU: 7.9% | Mem: 2484.6 MB

[Batch 1/100] Complete in 1.22s | Suspicious: 497 | Total Attacks Logged: 497 | CPU: 7.9% | Mem: 2484.6 MB

2026-03-20 17:38:02 - INFO - [Batch 1/100] Complete in 1.22s | S


🏁 SIMULATION COMPLETE

Processing Statistics:
  Total Packets   : 50,000
  Throughput      : 394.09 packets/s
  Avg Latency     : 2.40ms
  L1 Suspicious   : 49,866
  L2 Attacks      : 49,866

📊 ALERT SUMMARY

By Severity Level:
  CRITICAL:   48,663 alerts
      HIGH:       22 alerts
    MEDIUM:      764 alerts
       LOW:      417 alerts

By Attack Type:
  [CRITICAL] UDP Flood                :   15,555
  [CRITICAL] ICMP Flood               :   13,894
  [CRITICAL] SYN Flood                :    8,949
  [CRITICAL] TCP Flood                :    8,511
  [CRITICAL] MQTT Connect Flood       :    1,289
  [  MEDIUM] Reconnaissance           :      684
  [CRITICAL] MQTT Publish Flood       :      465
  [     LOW] Ambiguous                :      417
  [  MEDIUM] ARP Spoofing             :       77
  [    HIGH] MQTT Malformed           :       22
  [  MEDIUM] Recon (VulnScan)         :        3

🖥️  RESOURCE USAGE REPORT

  Logical CPU Cores     : 12
  Peak CPU Utilization  : 46.8%  (of total cap